## Greedy Policy

#### Download necessary modules

In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

#### Predicting hourly demand based on 2018 data 

In [32]:
# Load cleaned data (2018)
rideData2018 = pd.read_csv("../data/cleanData/df2_2018(clean_parks_metadate).csv")
rideData2018.head()

,date,wdw_ticket_season,dayofweek,dayofyear,weekofyear,monthofyear,year,season,holiday,wdwticketseason,...,hsfirewks,akprdday,akprddt1,akprddt2,akprddn,akfiren,akshwngt,akshwnt1,akshwnt2,akshwnn
0,2018-01-01,peak,2,0,0,1,2018,CHRISTMAS PEAK,1,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
1,2018-01-02,peak,3,1,0,1,2018,CHRISTMAS,0,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
2,2018-01-03,peak,4,2,0,1,2018,CHRISTMAS,0,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
3,2018-01-04,regular,5,3,0,1,2018,CHRISTMAS,0,regular,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
4,2018-01-05,regular,6,4,0,1,2018,CHRISTMAS,0,regular,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light


In [33]:
# Load waiting times data
waitTimes = pd.read_csv("../data/cleanData/animal_kingdom_df1(touringplans_2018).csv")
waitTimes.head()

,park_date,wait_hour,attraction_name,wait_minutes_posted_avg,attraction_duration,attraction_park,attraction_land,park_open,park_close,park_extra_magic_morning,park_extra_magic_evening,park_ticket_season,park_temperature_average,park_temperature_high,attraction_short_name
0,2018-01-01,8,DINOSAUR,15.000000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
1,2018-01-01,9,DINOSAUR,18.333333,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
2,2018-01-01,10,DINOSAUR,23.750000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
3,2018-01-01,11,DINOSAUR,24.000000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
4,2018-01-01,12,DINOSAUR,31.875000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR


In [34]:
# Fun Factor
# Fun factor "a" calculation
a_dict = {'Kanika': {'Speed': 10, 'Music': 4, 'Show': 5, 'Light': 5, 'Water': 8, '3D': 0},
          'Lydia': {'Speed': 6, 'Music': 3, 'Show': 1, 'Light': 7, 'Water': 1, '3D': 3},
          'Shawn': {'Speed': 7, 'Music': 8, 'Show': 0, 'Light': 1, 'Water': 6, '3D': 8},
          'Clarice': {'Speed': 4, 'Music': 5, 'Show': 10, 'Light': 2, 'Water': 3, '3D': 5},
          'Ethan': {'Speed': 5, 'Music': 6, 'Show': 2, 'Light': 1, 'Water': 4, '3D': 6},
          'Kevin': {'Speed': 6, 'Music': 7, 'Show': 1, 'Light': 0, 'Water': 5, '3D': 7},
          'Allison': {'Speed': 10, 'Music': 0, 'Show': 0, 'Light': 0, 'Water': 3, '3D': 2}}

# Load attributes of rides 
rideAttributes = pd.read_csv("../data/cleanData/animal_kingdom_ride_attributes.csv")

# Fun factor calculation
def calculate_fun_factor(person):
    ride_fun_factors = {}
    for ride in rideAttributes['Ride']:
        fun_factor = 0
        attributes = rideAttributes[rideAttributes['Ride'] == ride].iloc[0]
        fun_factor += (a_dict[person]['Speed'] * attributes['Speed'] +
                       a_dict[person]['Music'] * attributes['Music'] +
                       a_dict[person]['Show'] * attributes['Show'] +
                       a_dict[person]['Light'] * attributes['Light'] +
                       a_dict[person]['Water'] * attributes['Water'] +
                       a_dict[person]['3D'] * attributes['3D'])
        ride_fun_factors[ride] = float(fun_factor)
    return ride_fun_factors

In [35]:
calculate_fun_factor('Kanika')

{'DINOSAUR': 51.0,
 'Expedition Everest - Legend of the Forbidden Mountain': 62.0,
 'Avatar Flight of Passage': 78.0,
 'Kilimanjaro Safaris': 50.0,
 "Na'vi River Journey": 76.0}

In [36]:
# Create average wait time per attraction
average_wait_times = waitTimes.groupby(['wait_hour', 'attraction_name'])['wait_minutes_posted_avg'].mean().reset_index()
average_wait_times.columns = ['wait_hour', 'attraction_name', 'average_wait_time']
average_wait_times.head()

,wait_hour,attraction_name,average_wait_time
0,4,DINOSAUR,5.000000
1,4,Kilimanjaro Safaris,31.000000
2,5,Kilimanjaro Safaris,51.000000
3,6,Avatar Flight of Passage,65.803571
4,6,DINOSAUR,5.000000


In [37]:
# Attraction Duration for each ride
attraction_durations = waitTimes[['attraction_name', 'attraction_duration']].rename(columns={'attraction_name': 'Ride', 'attraction_duration': 'Duration'})
# Only keep unique rides
attraction_durations = attraction_durations.drop_duplicates(subset=['Ride'])
# Change Index to Ride
attraction_durations = attraction_durations.set_index('Ride')
attraction_durations.head()

,Duration
Ride,
DINOSAUR,3.5
Expedition Everest - Legend of the Forbidden Mountain,4.0
Avatar Flight of Passage,6.0
Kilimanjaro Safaris,20.0
Na'vi River Journey,5.0


In [38]:
def get_wait_time(ride, hour, average_wait_times):
    df = average_wait_times[
        (average_wait_times['attraction_name'] == ride) &
        (average_wait_times['wait_hour'] == hour)
    ]
    return float(df['average_wait_time'].iloc[0]) if not df.empty else 0.0

In [39]:
# Always-Greedy Heuristic 

def build_greedy(
    startTime = 8 * 60,     # 8 AM
    endTime   = 20 * 60,    # 8 PM
    alpha=0.8,
    beta=0.4,
    rideCountDecay = 0.3,
    person='Kanika',
    duration=attraction_durations,
    average_wait_times=average_wait_times,
    allow_repeats=True, 
    use_netgain=True, # True → (Fun − α·wait − β·ride), False → Fun / (α·wait+β·ride)
    verbose=False
    ):
    T = endTime - startTime
    t = 0
    rides = duration.index.tolist()
    ride_durations = duration['Duration'].astype(float).to_dict()
    fun_factors = calculate_fun_factor(person)

    remaining = set(rides)
    ride_counts = {r: 0 for r in rides}

    path, logs = [], []
    total_fun = total_wait = total_ride = 0.0
    step = 1

    while (remaining or allow_repeats) and t < T:
        current_hour = int((startTime + t) // 60)
        best_ride, best_score, best_wait, best_ride_time = None, -np.inf, None, None

        # candidates: remaining if not allowing repeats, all if repeats
        candidates = rides if allow_repeats else list(remaining)    
        
        for r in candidates:
            ride_time = float(ride_durations[r])
            wait_time = get_wait_time(r, current_hour, average_wait_times)
            if t + wait_time + ride_time > T:
                continue  # can't finish before closing

            net_fun = fun_factors[r] * (rideCountDecay ** ride_counts[r])

            if use_netgain:
                score = net_fun - alpha * wait_time - beta * ride_time
            else:
                score = net_fun / (alpha * wait_time + beta * ride_time + 1e-9)

            if score > best_score:
                best_ride, best_score = r, score
                best_wait, best_ride_time = wait_time, ride_time

        if best_ride is None:
            if verbose:
                print("No feasible rides left.")
            break

        # record choice
        start_clock = startTime + t
        finish_clock = start_clock + best_wait + best_ride_time
        path.append(best_ride)
        total_fun += fun_factors[best_ride]
        total_wait += best_wait
        total_ride += best_ride_time

        # just to see what greedy algorithm did
        logs.append({
            "step": step,
            "ride": best_ride,
            "hour": current_hour,
            "fun": fun_factors[best_ride],
            "wait_min": best_wait,
            "ride_min": best_ride_time,
            "score": best_score,
            "start_time_min": start_clock,
            "finish_time_min": finish_clock
        })

        # update state
        t += int(round(best_wait + best_ride_time))
        ride_counts[best_ride] += 1
        if not allow_repeats:
            remaining.discard(best_ride)
        step += 1
    
    return path

In [40]:
print(build_greedy(person ='Kanika'))
print(build_greedy(person ='Clarice'))
print(build_greedy(person ='Lydia'))
print(build_greedy(person ='Shawn'))
print(build_greedy(person ='Allison'))

['Expedition Everest - Legend of the Forbidden Mountain', "Na'vi River Journey", 'DINOSAUR', 'Kilimanjaro Safaris', 'DINOSAUR', 'Expedition Everest - Legend of the Forbidden Mountain', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'Avatar Flight of Passage', 'Kilimanjaro Safaris', 'DINOSAUR', 'DINOSAUR', 'Expedition Everest - Legend of the Forbidden Mountain', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR']
['DINOSAUR', "Na'vi River Journey", 'Expedition Everest - Legend of the Forbidden Mountain', 'Kilimanjaro Safaris', 'DINOSAUR', 'Expedition Everest - Legend of the Forbidden Mountain', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'Avatar Flight of Passage', 'Kilimanjaro Safaris', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR']
['Expedition Everest - Legend of the Forbidden Mountain', 'DINOSAUR', "Na'vi River Journey", 'Kilimanjaro Safaris', 'DINOSAUR', 'Expedition Everest - Legend of the Forbidden Mountain', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR', 'DINOSAUR'